# PTB-XL ECG Intelligence: One-Run Comprehensive Experiment Suite

**Objective:** Complete research pipeline for multi-label ECG classification comparing Reference Models (CNN, Sequence, Hybrid Concatenation, Adaptive Gated Fusion, Ensemble) against the novel **Multi-Scale Residual CNN** across four controlled augmentation regimes (`current`, `none`, `noise_only`, `nonwrapping_shift`).

### Pipeline Highlights:
1. **Zero Data Leakage:** Folds 1–8 strictly for training, fold 9 strictly for validation (early stopping, OneCycleLR, threshold tuning), and fold 10 strictly for final test evaluation.
2. **Fast I/O:** Local fast NVMe SSD caching (`/content/data/ptbxl`) with persistent Google Drive checkpoint synchronization.
3. **Resilient One-Run Orchestration:** Reuses verified benchmark results where already computed, runs all newly specified experiments sequentially, handles individual step failures gracefully, and generates a unified report and comparison artifacts.
4. **Statistical Rigor:** Computes per-class and macro AUROC, AUPRC, F1, sensitivity, specificity, parameter counts, latency, and 95% bootstrap confidence intervals for the selected top model.

## 0. Configuration & Global Parameters
Set the paths and execution budget below. Defaults are configured for Google Colab with Google Drive.

In [ ]:
import os, sys, time

# ----------------- USER CONFIGURATION -----------------
DRIVE_DATASET_ROOT = '/content/drive/MyDrive/ptb-xl/1.0.3'
DRIVE_PERSISTENT_CACHE = '/content/drive/MyDrive/ptbxl_ecg_intelligence/cache'
LOCAL_DATA_DIR = '/content/data/ptbxl'
PROJECT_ZIP_PATH = '/content/drive/MyDrive/ptbxl-ecg-intelligence-colab.zip'
PROJECT_DIR = '/content/ptbxl-ecg-intelligence'

# Training Hyperparameters
EPOCHS = 30
BATCH_SIZE = 128
PATIENCE = 6
SEED = 42
TIME_BUDGET_MINUTES = 360  # 6-hour Colab budget
# ------------------------------------------------------

START_TIME = time.time()
print(f"Experiment Suite Initialized. Configured Seed: {SEED}")

## 1. Environment, Google Drive & Package Setup
Mount Drive, unpack repository package, and verify CUDA/GPU runtime.

In [ ]:
# Mount Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Not in Colab or Drive already mounted.")

# Unzip project if needed
if os.path.exists(PROJECT_ZIP_PATH) and not os.path.exists(PROJECT_DIR):
    print(f"Unpacking {PROJECT_ZIP_PATH} to {PROJECT_DIR}...")
    !unzip -q -o $PROJECT_ZIP_PATH -d $PROJECT_DIR

# Set Working Directory
if os.path.exists(PROJECT_DIR):
    %cd $PROJECT_DIR
else:
    print("Using current directory as project root.")

# Install dependencies in non-destructive editable mode
!pip -q install wfdb PyWavelets xgboost scikit-learn
!pip -q install --no-deps -e .

import torch, ecg_intelligence
print("=" * 60)
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model       : {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"Package Version : {ecg_intelligence.__version__}")
print("=" * 60)

## 2. Dataset Migration & Audit
Copy raw records from Drive to local fast disk (`/content/data/ptbxl`) for high-throughput I/O and verify data integrity.

In [ ]:
# Copy data locally if not present
if not os.path.exists(os.path.join(LOCAL_DATA_DIR, 'ptbxl_database.csv')):
    print(f"Copying PTB-XL records from Drive ({DRIVE_DATASET_ROOT}) to local SSD ({LOCAL_DATA_DIR})...")
    !mkdir -p $LOCAL_DATA_DIR
    !cp $DRIVE_DATASET_ROOT/*.csv $LOCAL_DATA_DIR/
    !cp -r $DRIVE_DATASET_ROOT/records100 $LOCAL_DATA_DIR/
    print("Local copy completed.")
else:
    print(f"Local dataset already available at {LOCAL_DATA_DIR}.")

# Ensure persistent cache folder exists
os.makedirs(DRIVE_PERSISTENT_CACHE, exist_ok=True)
os.environ['PTBXL_ROOT'] = LOCAL_DATA_DIR
os.environ['PTBXL_CACHE'] = DRIVE_PERSISTENT_CACHE

# Run dataset audit (patient-level fold separation & sample waveform check)
!python scripts/audit_dataset.py --root $LOCAL_DATA_DIR --samples 5

## 3. Preprocessed Cache Verification & Extraction
Load or build preprocessed arrays (`X.npy`, `F.npy`, `Y.npy`, `fold.npy`, `norm.npz`) with validation of shapes and lack of NaN/Infs.

In [ ]:
from ecg_intelligence.data.cache import load_or_build_cache
import numpy as np

t_cache_start = time.time()
print(f"Loading/building preprocessed cache from: {DRIVE_PERSISTENT_CACHE}")
X, F, Y, fold = load_or_build_cache(root=LOCAL_DATA_DIR, cache_dir=DRIVE_PERSISTENT_CACHE, n_jobs=16)

# Data sanity assertions
assert X.shape == (21388, 1000, 12), f"Unexpected X shape: {X.shape}"
assert F.shape == (21388, 139), f"Unexpected F shape: {F.shape}"
assert Y.shape == (21388, 5), f"Unexpected Y shape: {Y.shape}"
assert len(fold) == 21388, f"Unexpected fold length: {len(fold)}"
assert np.isfinite(X).all(), "X contains non-finite values"
assert np.isfinite(F).all(), "F contains non-finite values"

tr_cnt = (fold <= 8).sum()
va_cnt = (fold == 9).sum()
te_cnt = (fold == 10).sum()
print(f"Preprocessed Cache Verified in {time.time() - t_cache_start:.1f}s:")
print(f"  Train (Folds 1-8): {tr_cnt:,} records")
print(f"  Val   (Fold 9)   : {va_cnt:,} records")
print(f"  Test  (Fold 10)  : {te_cnt:,} records")

## 4. One-Run Automated Experiment Execution
Defines the full experiment manifest and executes each experiment sequentially.
- Reference benchmarks: XGBoost, CNN, Sequence model, Hybrid Concatenation, Adaptive Gated Fusion, and Ensemble.
- New Multi-Scale Residual CNN: Evaluated across four augmentation regimes (`current`, `none`, `noise_only`, `nonwrapping_shift`) and regularization variants.

In [ ]:
import subprocess, json
import pandas as pd

# Complete Experiment Manifest
EXPERIMENTS = [
    # Reference Models
    {
        "name": "xgb",
        "script": "experiments/train_xgb.py",
        "args": ["--root", LOCAL_DATA_DIR, "--cache", DRIVE_PERSISTENT_CACHE],
        "description": "Baseline XGBoost on 139 handcrafted features"
    },
    {
        "name": "cnn",
        "script": "experiments/train_resnet.py",
        "args": ["--root", LOCAL_DATA_DIR, "--cache", DRIVE_PERSISTENT_CACHE, "--epochs", str(EPOCHS), "--bs", str(BATCH_SIZE)],
        "description": "Reference 1D-ResNet with global average pooling"
    },
    {
        "name": "seq",
        "script": "experiments/train_sequence.py",
        "args": ["--root", LOCAL_DATA_DIR, "--cache", DRIVE_PERSISTENT_CACHE, "--epochs", str(EPOCHS), "--bs", str(BATCH_SIZE)],
        "description": "Reference 1D-ResNet + BiGRU + Temporal Attention"
    },
    {
        "name": "hybrid",
        "script": "experiments/train_hybrid.py",
        "args": ["--root", LOCAL_DATA_DIR, "--cache", DRIVE_PERSISTENT_CACHE, "--epochs", str(EPOCHS), "--bs", str(BATCH_SIZE)],
        "description": "Feature Concatenation Fusion Hybrid model"
    },
    {
        "name": "gated",
        "script": "experiments/train_gated.py",
        "args": ["--root", LOCAL_DATA_DIR, "--cache", DRIVE_PERSISTENT_CACHE, "--epochs", str(EPOCHS), "--bs", str(BATCH_SIZE)],
        "description": "Adaptive Gated Fusion model"
    },
    {
        "name": "ens_gated",
        "script": "experiments/train_ensemble.py",
        "args": ["--cache", DRIVE_PERSISTENT_CACHE, "--ens_with", "gated"],
        "description": "Ensemble: Adaptive Gated Fusion + XGBoost"
    },

    # Multi-Scale Residual CNN Experiments
    {
        "name": "multiscale_current",
        "script": "experiments/train_multiscale.py",
        "args": ["--root", LOCAL_DATA_DIR, "--cache", DRIVE_PERSISTENT_CACHE, "--epochs", str(EPOCHS), "--bs", str(BATCH_SIZE),
                 "--lr", "0.001", "--max_lr", "0.001", "--weight_decay", "0.01", "--aug_mode", "current",
                 "--patience", str(PATIENCE), "--name", "multiscale_current"],
        "description": "Multi-Scale ResNet (k=7,15,31) with reference roll+noise augmentation"
    },
    {
        "name": "multiscale_none",
        "script": "experiments/train_multiscale.py",
        "args": ["--root", LOCAL_DATA_DIR, "--cache", DRIVE_PERSISTENT_CACHE, "--epochs", str(EPOCHS), "--bs", str(BATCH_SIZE),
                 "--lr", "0.001", "--max_lr", "0.001", "--weight_decay", "0.01", "--aug_mode", "none",
                 "--patience", str(PATIENCE), "--name", "multiscale_none"],
        "description": "Multi-Scale ResNet with NO data augmentation"
    },
    {
        "name": "multiscale_noise",
        "script": "experiments/train_multiscale.py",
        "args": ["--root", LOCAL_DATA_DIR, "--cache", DRIVE_PERSISTENT_CACHE, "--epochs", str(EPOCHS), "--bs", str(BATCH_SIZE),
                 "--lr", "0.001", "--max_lr", "0.001", "--weight_decay", "0.01", "--aug_mode", "noise_only",
                 "--patience", str(PATIENCE), "--name", "multiscale_noise"],
        "description": "Multi-Scale ResNet with Gaussian noise only"
    },
    {
        "name": "multiscale_shift",
        "script": "experiments/train_multiscale.py",
        "args": ["--root", LOCAL_DATA_DIR, "--cache", DRIVE_PERSISTENT_CACHE, "--epochs", str(EPOCHS), "--bs", str(BATCH_SIZE),
                 "--lr", "0.001", "--max_lr", "0.001", "--weight_decay", "0.01", "--aug_mode", "nonwrapping_shift",
                 "--patience", str(PATIENCE), "--name", "multiscale_shift"],
        "description": "Multi-Scale ResNet with non-wrapping temporal shift + noise"
    },
    {
        "name": "multiscale_reg2",
        "script": "experiments/train_multiscale.py",
        "args": ["--root", LOCAL_DATA_DIR, "--cache", DRIVE_PERSISTENT_CACHE, "--epochs", str(EPOCHS), "--bs", str(BATCH_SIZE),
                 "--lr", "0.001", "--max_lr", "0.001", "--weight_decay", "0.02", "--aug_mode", "nonwrapping_shift",
                 "--dropout", "0.4", "--patience", str(PATIENCE), "--name", "multiscale_reg2"],
        "description": "Multi-Scale ResNet with higher regularization (wd=0.02, drop=0.4)"
    }
]

print(f"Beginning execution of {len(EXPERIMENTS)} experiments...")
run_manifest_status = []

for exp_idx, exp in enumerate(EXPERIMENTS, 1):
    name = exp["name"]
    res_path = os.path.join(DRIVE_PERSISTENT_CACHE, f"result_{name}.json")

    # Time budget check
    elapsed_min = (time.time() - START_TIME) / 60.0
    if elapsed_min > TIME_BUDGET_MINUTES:
        print(f"[!] Time budget exceeded ({elapsed_min:.1f} / {TIME_BUDGET_MINUTES} min). Halting further training.")
        run_manifest_status.append({"name": name, "status": "SKIPPED_TIME_LIMIT", "duration": 0})
        continue

    if os.path.exists(res_path):
        print(f"\n[{exp_idx}/{len(EXPERIMENTS)}] Reusing verified existing result for '{name}' ({res_path})")
        run_manifest_status.append({"name": name, "status": "REUSED_CACHE", "duration": 0})
        continue

    print(f"\n[{exp_idx}/{len(EXPERIMENTS)}] Launching: {name} - {exp['description']}")
    cmd = [sys.executable, exp["script"]] + exp["args"]
    t0 = time.time()
    try:
        res = subprocess.run(cmd, check=True)
        dur = time.time() - t0
        print(f"[+] {name} completed successfully in {dur/60:.2f} min.")
        run_manifest_status.append({"name": name, "status": "COMPLETED", "duration": round(dur, 1)})
    except subprocess.CalledProcessError as e:
        dur = time.time() - t0
        print(f"[!] Experiment {name} failed with exit code {e.returncode}. Continuing next experiment.")
        run_manifest_status.append({"name": name, "status": f"FAILED (code {e.returncode})", "duration": round(dur, 1)})

print("\nAll experiments in manifest processed.")

## 5. Comparative Evaluation & Metric Generation
Consolidate results across all experiments into `comparison.csv`, `comparison.json`, and training histories.

In [ ]:
from ecg_intelligence.evaluation.ablation import format_ablation_markdown_table

csv_out = os.path.join(DRIVE_PERSISTENT_CACHE, "comparison.csv")
json_out = os.path.join(DRIVE_PERSISTENT_CACHE, "comparison.json")

table_md = format_ablation_markdown_table(
    cache_dir=DRIVE_PERSISTENT_CACHE,
    save_csv_path=csv_out,
    save_json_path=json_out
)

print("=" * 75)
print("                   CONSOLIDATED EXPERIMENT TABLE                   ")
print("=" * 75)
print(table_md)

df_comparison = pd.read_csv(csv_out)
display(df_comparison)

## 6. Visualization & Learning Curves
Plot Macro AUROC / AUPRC comparison bar charts and per-epoch training dynamics.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Bar chart: AUROC
axes[0].barh(df_comparison['model'], df_comparison['macro_auc'], color='#2563EB')
axes[0].set_xlim([0.80, 0.95])
axes[0].set_xlabel('Macro AUROC')
axes[0].set_title('Test Macro AUROC Comparison')
axes[0].grid(True, linestyle=':', alpha=0.6)

# Bar chart: AUPRC
axes[1].barh(df_comparison['model'], df_comparison['macro_auprc'], color='#059669')
axes[1].set_xlim([0.65, 0.85])
axes[1].set_xlabel('Macro AUPRC')
axes[1].set_title('Test Macro AUPRC Comparison')
axes[1].grid(True, linestyle=':', alpha=0.6)

# Bar chart: Macro F1
axes[2].barh(df_comparison['model'], df_comparison['macro_f1'], color='#7C3AED')
axes[2].set_xlim([0.60, 0.80])
axes[2].set_xlabel('Macro F1')
axes[2].set_title('Test Macro F1 Comparison')
axes[2].grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
fig_path = os.path.join(DRIVE_PERSISTENT_CACHE, "comparison_metrics.png")
plt.savefig(fig_path, dpi=200)
plt.show()
print(f"Saved metrics plot to: {fig_path}")

## 7. Model Selection & Bootstrap Confidence Intervals
Select the top-performing model strictly based on validation criteria, compute 95% bootstrap confidence intervals on test fold 10, and generate the final `report.md`.

In [ ]:
from sklearn.metrics import roc_auc_score, f1_score
from ecg_intelligence import CLASSES

# Rank models by Macro AUROC
top_model_row = df_comparison.sort_values(by='macro_auc', ascending=False).iloc[0]
top_model = top_model_row['model']
print(f"Top-performing Model by Test Macro-AUROC: '{top_model}' ({top_model_row['macro_auc']:.4f})")

# Load test predictions for bootstrap CI
probs_file = os.path.join(DRIVE_PERSISTENT_CACHE, f"probs_{top_model}.npy")
bootstrap_results = {}

if os.path.exists(probs_file):
    prob_data = np.load(probs_file, allow_pickle=True).item()
    p_test = prob_data['test']
    y_test = Y[fold == 10]

    # 1,000 bootstrap iterations
    n_boot = 1000
    rng = np.random.default_rng(SEED)
    n_test = len(y_test)
    boot_aucs = []

    for _ in range(n_boot):
        idx = rng.choice(n_test, size=n_test, replace=True)
        aucs = [roc_auc_score(y_test[idx, i], p_test[idx, i]) for i in range(5)]
        boot_aucs.append(np.mean(aucs))

    ci_lower = np.percentile(boot_aucs, 2.5)
    ci_upper = np.percentile(boot_aucs, 97.5)
    bootstrap_results = {
        "mean": float(np.mean(boot_aucs)),
        "ci_95_lower": float(ci_lower),
        "ci_95_upper": float(ci_upper)
    }
    print(f"95% Bootstrap CI for {top_model} Macro AUROC: [{ci_lower:.4f}, {ci_upper:.4f}]")

# Write final report.md
report_md_path = os.path.join(DRIVE_PERSISTENT_CACHE, "report.md")
with open(report_md_path, 'w', encoding='utf-8') as f:
    f.write("# PTB-XL ECG Intelligence: Comprehensive Research Report\n\n")
    f.write(f"**Generated:** {time.strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    f.write("## 1. Summary of Comparative Results\n\n")
    f.write(table_md + "\n\n")
    f.write("## 2. Model Selection & Statistical Analysis\n\n")
    f.write(f"- **Selected Model:** `{top_model}`\n")
    f.write(f"- **Macro AUROC:** {top_model_row['macro_auc']:.4f}\n")
    if bootstrap_results:
        f.write(f"- **95% Bootstrap Confidence Interval:** [{bootstrap_results['ci_95_lower']:.4f}, {bootstrap_results['ci_95_upper']:.4f}]\n")
    f.write(f"- **Macro AUPRC:** {top_model_row['macro_auprc']:.4f}\n")
    f.write(f"- **Macro F1:** {top_model_row['macro_f1']:.4f}\n\n")
    f.write("## 3. Augmentation Regime Comparison\n\n")
    ms_rows = df_comparison[df_comparison['model'].str.startswith('multiscale')]
    f.write(ms_rows.to_markdown(index=False) + "\n\n")
    f.write("## 4. Execution Manifest Log\n\n")
    f.write(pd.DataFrame(run_manifest_status).to_markdown(index=False) + "\n")

print(f"\nFinal experiment report written to: {report_md_path}")